In [1]:
import pickle
import pandas as pd

In [2]:
with open("processed_data_pkl/imputed_training_data.pkl", "rb") as f:
    traffic_dic = pickle.load(f)

with open("processed_data_pkl/weather_global_2014_2025.pkl", "rb") as f:
    weather = pickle.load(f)

air_qual = pd.read_csv("online_data/air_qual/air_qual.csv")

In [3]:
air_qual.drop(columns=["aerosol_optical_depth ()", "dust (μg/m³)"], inplace=True)
air_qual.dropna(inplace=True)
air_qual.reset_index(inplace=True, drop=True)

In [4]:
air_qual["time"] = pd.to_datetime(air_qual["time"], errors="coerce")
weather["timestamp"] = pd.to_datetime(weather["timestamp"], errors="coerce")

weather = weather[weather["timestamp"].isin(air_qual["time"])]
weather.reset_index(inplace=True, drop=True)

# rf base line with lag

In [5]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from sklearn.model_selection import train_test_split
import numpy as np
import seaborn as sns

In [6]:
air_qual_no_time = air_qual.drop(columns=["time"]).copy()

In [7]:
df = air_qual.copy()

# Create lag features
for col in [
    "pm10 (μg/m³)",
    "pm2_5 (μg/m³)",
    "nitrogen_dioxide (μg/m³)",
    "sulphur_dioxide (μg/m³)",
    "carbon_monoxide (μg/m³)",
    "ozone (μg/m³)"
]:
    df[f"{col}_lag1"] = df[col].shift(1)
    df[f"{col}_lag3"] = df[col].shift(3)

# Target is current PM2.5
y = df["pm2_5 (μg/m³)"]

# Keep ONLY lagged features
x = df[[c for c in df.columns if "_lag" in c]]

# Remove rows with NaNs from shifting
mask = x.notna().all(axis=1)
x = x[mask]
y = y[mask]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, shuffle=False)

rf = RandomForestRegressor(random_state=42)
rf.fit(x_train, y_train)

pred = rf.predict(x_test)

In [ ]:
pred = rf.predict(x_test)
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)

print("MAE", mae)
print("RMSE", np.sqrt(mean_squared_error(y_test, pred)))
print("R2", r2)

MAE 0.613991388160146
RMSE 0.9652411813103557
R2 0.9701230428704614


# lstm

In [ ]:
features = [
    "pm10 (μg/m³)",
    "nitrogen_dioxide (μg/m³)",
    "sulphur_dioxide (μg/m³)",
    "carbon_monoxide (μg/m³)",
    "ozone (μg/m³)"
]

scaler = MinMaxScaler()

scaled = scaler.fit_transform(air_qual[features + ["pm2_5 (μg/m³)"]])

In [ ]:
lookback = 24
forecast_steps = 5 

x = []
y = []

for i in range(lookback, len(scaled) - forecast_steps + 1):
    x.append(scaled[i-lookback:i, :-1])
    y.append(scaled[i:i+forecast_steps, -1])

x = np.array(x)
y = np.array(y)

In [ ]:
# Train/Test Split
split = int(len(x) * 0.8)
x_train, x_test = x[:split], x[split:]
y_train, y_test = y[:split], y[split:]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(128, input_shape=(x_train.shape[1], x_train.shape[2]), return_sequences=True),
    Dropout(0.2),
    
    LSTM(64, return_sequences=False), 
    Dropout(0.2),
    
    Dense(32, activation="relu"),
    Dense(forecast_steps) 
])

model.compile(optimizer="adam", loss="mse")

model.fit(x_train, y_train, epochs=30, batch_size=64)

Epoch 1/30
 115/1096 ━━━━━━━━━━━━━━━━━━━━ 33s 34ms/step - loss: 0.0067

In [ ]:
pred = model.predict(x_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

548/548 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
MAE: 0.02269795559154168
RMSE: 0.032125847082575876
R²: 0.8143634298445044


In [ ]:
import plotly.graph_objects as go

target_min = scaler.min_[-1]
target_scale = scaler.scale_[-1]

y_test_true_rescaled = (y_test - target_min) / target_scale
y_test_pred_rescaled = (pred - target_min) / target_scale

test_start_idx = split + lookback
pm25_raw = air_qual["pm2_5 (μg/m³)"].values

# Timeline Arrays for X-axis positioning
history_x = list(range(lookback))
forecast_x = list(range(lookback, lookback + forecast_steps))

fig = go.Figure()

sample_idx = 0
global_start = test_start_idx + sample_idx

fig.add_trace(go.Scatter(
    x=history_x, 
    y=pm25_raw[global_start - lookback : global_start],
    mode='lines+markers', name='Historical PM2.5', line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=forecast_x, 
    y=y_test_true_rescaled[sample_idx],
    mode='lines+markers', name='Actual Future', line=dict(color='green', dash='dash')
))

fig.add_trace(go.Scatter(
    x=forecast_x, 
    y=y_test_pred_rescaled[sample_idx],
    mode='lines+markers', name='Predicted Forecast', line=dict(color='red')
))

# 2. Build the dropdown menus for the first 50 test samples
num_samples_to_show = min(50, len(x_test))
buttons = []

for idx in range(num_samples_to_show):
    visibility = [False] * (num_samples_to_show * 3)
    visibility[idx * 3] = True
    visibility[idx * 3 + 1] = True
    visibility[idx * 3 + 2] = True
    
    button = dict(
        label=f"Sample {idx}",
        method="update",
        args=[
            {"visible": visibility},
            {"title": f"PM2.5 Timeline: 24h History + 5-Step Forecast (Sample {idx})"}
        ]
    )
    buttons.append(button)

# 3. Generate all remaining hidden traces for the selector upfront
for idx in range(1, num_samples_to_show):
    g_start = test_start_idx + idx
    fig.add_trace(go.Scatter(
        x=history_x, y=pm25_raw[g_start - lookback : g_start],
        mode='lines+markers', name='Historical PM2.5', line=dict(color='blue'), visible=False
    ))
    # Actual
    fig.add_trace(go.Scatter(
        x=forecast_x, y=y_test_true_rescaled[idx],
        mode='lines+markers', name='Actual Future', line=dict(color='green', dash='dash'), visible=False
    ))
    # Predicted
    fig.add_trace(go.Scatter(
        x=forecast_x, y=y_test_pred_rescaled[idx],
        mode='lines+markers', name='Predicted Forecast', line=dict(color='red'), visible=False
    ))

# 4. Final Layout Customization
fig.update_layout(
    updatemenus=[dict(active=0, buttons=buttons, direction="down", pad={"r": 10, "t": 10}, showactive=True, x=0.02, xanchor="left", y=1.15, yanchor="top")],
    title="PM2.5 Timeline: 24h History + 5-Step Forecast (Sample 0)",
    xaxis_title="Timeline (Hours)",
    yaxis_title="PM2.5 Concentration (μg/m³)",
    height=600,
    showlegend=True
)

fig.add_vline(x=23.5, line_width=2, line_dash="dash", line_color="gray")

fig.show()